# Ramp Resource LP — EFHK Validation Notebook

Validates all nine user stories against the EFHK reference dataset (`data/finavia_flights_efhk_20260327.csv`).

| Stage | Function | User story |
|-------|----------|------------|
| 1 | `compute_demand()` | US1–US3, US6–US8 |
| 2 | `schedule_shifts()` | US4–US5 |
| Analysis | `identify_bottlenecks()` | US9 |
| Analysis | `comparison_report()` | FR-008 |

In [ ]:
import sys
sys.path.insert(0, '../..')  # ensure src/ is importable from notebooks/planning/

from src.utils import load_efhk
from src.lp import (
    compute_demand, schedule_shifts,
    identify_bottlenecks, comparison_report,
    AircraftType,
)

## 1  Load EFHK Dataset

In [ ]:
DATA = '../../data/finavia_flights_efhk_20260327.csv'

scheduled, movements = load_efhk(DATA)
print(f'Scheduled slots : {len(scheduled)}')
print(f'Movement records: {len(movements) if movements else 0}')
print()
for slot in scheduled:
    print(f'  Hour {slot.hour:02d}:00 | arr {dict(slot.arrival_counts)} | dep {dict(slot.departure_counts)}')

## 2  Stage 1 — Demand Curve

In [ ]:
# US8: use tau_movements for tolerance-window reclassification (FR-011)
demand = compute_demand(scheduled, tau_movements=movements)

print(f'Feasible : {demand.feasible}')
print(f'Infeasible slots: {demand.infeasible_slots}')
print()
print(f'{"Hour":<6} {"Demand":>7} {"Arr":>5} {"Dep":>5}')
print('-' * 25)
for h, d, a, dp in zip(
    demand.operating_hours,
    demand.demand_curve,
    demand.arrival_demand_curve,
    demand.departure_demand_curve,
):
    print(f'{h:02d}:00  {d:>7}  {a:>5}  {dp:>5}')

## 3  Stage 2 — Minimum Shift Schedule

In [ ]:
schedule = schedule_shifts(demand)

print(f'Daily headcount  : {schedule.daily_headcount}')
print(f'Coverage satisfied: {schedule.coverage_satisfied}')
print(f'Coverage shortfalls: {schedule.coverage_shortfalls}')
print()
print(f'{"Start hour":<12} {"Shifts (rounded)":>16}')
print('-' * 30)
for h, n in sorted(schedule.shift_starts_rounded.items()):
    if n > 0:
        print(f'{h:02d}:00        {n:>16}')

## 4  Bottleneck Hours (US9)

In [ ]:
bottlenecks = identify_bottlenecks(demand, schedule)

if bottlenecks.bottleneck_hours:
    print('Bottleneck hours (active workers == demand):')
    for h in bottlenecks.bottleneck_hours:
        print(f'  {h:02d}:00  demand={bottlenecks.demand_at_bottleneck[h]}')
else:
    print('No bottleneck hours — all slots have surplus workers.')

## 5  Scheduled vs Tau Comparison (FR-008)

In [ ]:
tau_slots, _ = load_efhk(DATA, use_tau_times=True, extract_tau=False)
report = comparison_report(scheduled, tau_slots)

print(f'Arrival gap   : {report.arrival_gap_pct_total:+.1f}%')
print(f'Departure gap : {report.departure_gap_pct_total:+.1f}%')
print()
print(f'{"Hour":<6} {"S-Arr":>6} {"T-Arr":>6} {"Gap":>6} | {"S-Dep":>6} {"T-Dep":>6} {"Gap":>6}')
print('-' * 50)
for i, h in enumerate(report.hours):
    print(
        f'{h:02d}:00  '
        f'{report.scheduled_arrival_demand[i]:>6}  '
        f'{report.tau_arrival_demand[i]:>6}  '
        f'{report.arrival_gap_absolute[i]:>+6}  | '
        f'{report.scheduled_departure_demand[i]:>6}  '
        f'{report.tau_departure_demand[i]:>6}  '
        f'{report.departure_gap_absolute[i]:>+6}'
    )